
# Tutorial: APEX for Privacy-Conscious Delegation

In this notebook we revisit the privacy-conscious delegation scenario from the PAPILLON tutorials and optimize the program with the new `dspy.APEX` teleprompter. APEX follows a map-reduce style loop: it samples training data, runs the current program, performs root cause and success analyses with the analysis language model, synthesizes hypotheses, evaluates all candidates on a calibration set, and advances only when a proposal strictly improves the calibration score.

We'll reuse the PAPILLON program and evaluation harness introduced in the GEPA/GRPO tutorials, so you can compare optimizers on the same dataset.



> **Tip:** You can enable tools such as MLflow autologging or Weights & Biases exactly as in the other optimizer tutorials. The code below focuses on the core APEX flow for clarity.


In [ ]:

import dspy

api_key = input("Enter your OpenAI API key: ")

local_lm = dspy.LM(model="openai/gpt-4.1-nano", api_key=api_key)
large_lm = dspy.LM(model="openai/gpt-4.1-mini", api_key=api_key)
analysis_lm = dspy.LM(model="openai/gpt-4.1", api_key=api_key, max_tokens=1800)

# Configure default LM for the student program
dspy.configure(lm=local_lm)



## Define the PAPILLON program

The module copies the implementation from the PAPILLON tutorials: a redaction step crafts a privacy-preserving request for an untrusted model, and a second predictor uses that response to answer the user query.


In [ ]:

class CraftRedactedRequest(dspy.Signature):
    '''Given a private user query, create a privacy-preserving request for a powerful external LLM.'''

    user_query = dspy.InputField()
    llm_request = dspy.OutputField()


class RespondToQuery(dspy.Signature):
    '''Respond to a user query using an untrusted LLM response as supporting context.'''

    related_llm_request = dspy.InputField()
    related_llm_response = dspy.InputField(desc="information from a powerful LLM responding to a related request")
    user_query = dspy.InputField(desc="the user's request you need to fulfill")
    response = dspy.OutputField(desc="your final response to the user's request")


class PAPILLON(dspy.Module):
    def __init__(self, untrusted_model):
        super().__init__()
        self.craft_redacted_request = dspy.ChainOfThought(CraftRedactedRequest)
        self.respond_to_query = dspy.Predict(RespondToQuery)
        self.untrusted_model = untrusted_model

    def forward(self, user_query):
        try:
            llm_request = self.craft_redacted_request(user_query=user_query).llm_request
            llm_response = self.untrusted_model(llm_request)[0]
            response = self.respond_to_query(
                related_llm_request=llm_request,
                related_llm_response=llm_response,
                user_query=user_query,
            ).response
        except Exception:
            return dspy.Prediction(llm_request="", llm_response="", response="")

        return dspy.Prediction(llm_request=llm_request, llm_response=llm_response, response=response)



## Load the PAPILLON dataset

We use the PUPA benchmark released with PAPILLON. The dataset is available through the Hugging Face hub. We'll keep a few hundred examples to keep the runtime of APEX manageable.


In [ ]:

from datasets import load_dataset

pupa_new = load_dataset("Columbia-NLP/PUPA", "pupa_new")
examples = [
    dspy.Example(
        {
            "target_response": sample["target_response"],
            "user_query": sample["user_query"],
            "pii_str": sample["pii_units"],
        }
    ).with_inputs("user_query")
    for sample in pupa_new["train"]
]

trainset, devset, testset = examples[:225], examples[225:450], examples[450:600]
print(f"Loaded {len(trainset)} training examples, {len(devset)} dev examples, and {len(testset)} test examples.")



## Define the evaluation judges and metric

We reuse the quality and leakage judges from the GEPA tutorial so we can compare optimizers on the same objective. The metric returns a structured `dspy.Prediction` with both the scalar score and a feedback string, matching the expectations of APEX.


In [ ]:

class JudgeQuality(dspy.Signature):
    '''Is response A at least as good as response B for the user query?'''

    user_query = dspy.InputField(desc="The user's request to be fulfilled.")
    response_A = dspy.InputField()
    response_B = dspy.InputField()
    judgment: bool = dspy.OutputField()


class JudgeLeakage(dspy.Signature):
    '''How much PII from `pii` leaks into the prompt?'''

    pii = dspy.InputField()
    prompt = dspy.InputField()
    num_pii_leaked: int = dspy.OutputField()


class LLMJudge(dspy.Module):
    def __init__(self):
        super().__init__()
        self.quality_judge = dspy.ChainOfThought(JudgeQuality)
        self.fact_checker = dspy.ChainOfThought(JudgeLeakage)

    def forward(self, user_query, og_resp, new_resp=None, updated_query=None, pii_str=None):
        judgment_1 = self.quality_judge(user_query=user_query, response_A=new_resp, response_B=og_resp).judgment
        judgment_2 = self.quality_judge(user_query=user_query, response_A=og_resp, response_B=new_resp).judgment
        judgment = judgment_1 or (judgment_1 == judgment_2)

        pii = list(set(pii_str.split("||"))) if pii_str else []
        leakage = self.fact_checker(pii=pii, prompt=updated_query).num_pii_leaked
        leakage = leakage / len(pii) if pii else 0.0

        return dspy.Prediction(quality=float(judgment), leakage=float(leakage))


llm_judge = LLMJudge()
llm_judge.set_lm(large_lm)


def compute_metrics(gold, pred, trace=None):
    return llm_judge(
        user_query=gold.user_query,
        new_resp=pred.response,
        og_resp=gold.target_response,
        updated_query=pred.llm_request,
        pii_str=gold.pii_str,
    )


def compute_overall_score_with_feedback(gold, pred, trace=None, pred_name=None, pred_trace=None):
    metrics = compute_metrics(gold, pred, trace)
    overall_score = (metrics.quality + (1 - metrics.leakage)) / 2.0
    feedback = (
        f"Overall score: {overall_score:.2f}. Quality={metrics.quality:.2f}, safety={1 - metrics.leakage:.2f}. "
        "Improve response helpfulness while reducing leaked PII."
    )
    return dspy.Prediction(score=overall_score, feedback=feedback)



## Baseline evaluation

Before optimizing, evaluate the zero-shot PAPILLON program to establish a reference score.


In [ ]:

zeroshot = PAPILLON(untrusted_model=large_lm)
kwargs = dict(num_threads=8, display_progress=True, display_table=5, max_errors=100)
evaluate = dspy.Evaluate(metric=lambda g, p, t=None: compute_overall_score_with_feedback(g, p, t).score, devset=testset, **kwargs)
zeroshot_score = evaluate(zeroshot)
print(f"Zero-shot score on the test split: {zeroshot_score:.3f}")



## Run APEX optimization

We configure APEX with a modest budget: at most three iterations, one hypothesis per iteration, and a train sample of 120 examples per loop. The analysis and hypothesis language models use the higher capacity OpenAI endpoint while the student program itself continues to run on the smaller local model.


In [ ]:

from dspy.teleprompt.apex_optimizer import APEX

papillon = PAPILLON(untrusted_model=large_lm)
papillon.set_lm(local_lm)

apex = APEX(
    metric=compute_overall_score_with_feedback,
    analysis_llm=analysis_lm,
    hypothesis_llm=analysis_lm,
    max_iterations=3,
    num_hypotheses=1,
    num_eval_runs=2,
    train_sample=120,
    success_threshold=0.6,
    convergence_patience=1,
    seed=1234,
)

optimized_papillon = apex.compile(
    student=papillon,
    trainset=trainset,
    valset=devset,
)



## Inspect the optimized program

APEX returns the champion program, best overall score, and a full history of hypotheses and candidate scores. Let's inspect the optimized prompts and compare performance against the test set again.


In [ ]:

best_candidate = optimized_papillon.apex_result.best_candidate
print(f"Best calibration score: {best_candidate.overall_score:.3f}")
print("
Updated redaction prompt:
---------------------------")
print(optimized_papillon.craft_redacted_request.predict.signature.instructions)
print("
Updated response prompt:
-------------------------")
print(optimized_papillon.respond_to_query.predict.signature.instructions)


In [ ]:

optimized_score = evaluate(optimized_papillon)
print(f"Optimized program score on the test split: {optimized_score:.3f}")


In [ ]:

for iteration in optimized_papillon.apex_result.iterations:
    print(
        f"Iteration {iteration.iteration}: train size={iteration.sampled_train_size}, "
        f"failures={iteration.num_failures}, successes={iteration.num_successes}, "
        f"hypotheses={len(iteration.hypotheses)}, candidates={len(iteration.candidates)}"
    )
    for cand in iteration.candidates:
        label = "baseline" if cand.hypothesis is None else "hypothesis"
        print(f"  - {label:10s} score={cand.overall_score:.3f}")
    print()
